In [1]:
import pandas as pd


In [2]:
df = pd.read_excel('nadil_category_expenses.xlsx')

In [3]:
df.head()

,Date,Discription,Payments,Receipts,Balance,cleaned_particulars,Category,Cluster
0,2022-11-06,t ahirt OTHBNK T,6030.0,NaN,12454.64,t ahirt othbnk t,NADIL OTHBNK T,0
1,2022-11-06,010001088282101 OTHBNK T,3030.0,NaN,9424.64,010001088282101 othbnk t,OTHBNK T,1
2,2022-11-15,RIB/RMB SE.CH 20 IBMB Chg,25.0,NaN,9399.64,rib/rmb se.ch 20 ibmb charge,RIBRMB SECH IBMB CHARGE,2
3,2022-11-18,nadil Siriwardha MB SA TF,450.0,NaN,11537.14,nadil siriwardha mb sa tf,NADIL OTHBNK T,0
4,2022-12-24,nadil OTHBNK T,7530.0,NaN,27264.9,nadil othbnk t,NADIL OTHBNK T,0


In [4]:
print(df.head())

        Date                Discription  Payments  Receipts   Balance  \
0 2022-11-06           t ahirt OTHBNK T    6030.0       NaN  12454.64   
1 2022-11-06   010001088282101 OTHBNK T    3030.0       NaN   9424.64   
2 2022-11-15  RIB/RMB SE.CH 20 IBMB Chg      25.0       NaN   9399.64   
3 2022-11-18  nadil Siriwardha MB SA TF     450.0       NaN  11537.14   
4 2022-12-24             nadil OTHBNK T    7530.0       NaN   27264.9   

            cleaned_particulars                  Category  Cluster  
0              t ahirt othbnk t            NADIL OTHBNK T        0  
1      010001088282101 othbnk t                  OTHBNK T        1  
2  rib/rmb se.ch 20 ibmb charge  RIBRMB SECH  IBMB CHARGE        2  
3     nadil siriwardha mb sa tf            NADIL OTHBNK T        0  
4                nadil othbnk t            NADIL OTHBNK T        0  


In [7]:

# Assuming your Excel data is already loaded into a DataFrame 'df'
df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m-%d')
df['Payments'] = df['Payments'].astype(float)
df['Receipts'] = df['Receipts'].astype(float)
df['Balance'] = df['Balance'].astype(float)

# Create the full date range from the start to end date
start_date = df['Date'].min()
end_date = df['Date'].max()
all_dates = pd.date_range(start=start_date, end=end_date, freq='D')

# Create a list of all clusters
all_clusters = df['Cluster'].unique()

# Create a new DataFrame to hold the final results
final_df = pd.DataFrame(index=pd.MultiIndex.from_product([all_dates, all_clusters], names=['Date', 'Cluster']))

# Merge the original DataFrame with the new index
df_merged = pd.merge(final_df, df, on=['Date', 'Cluster'], how='left')

# Fill missing balances by carrying the previous balance value forward
df_merged['Balance'] = df_merged['Balance'].fillna(method='ffill')

# Group by Date and Cluster, summing up Payments and Receipts where applicable
df_merged = df_merged.groupby(['Date', 'Cluster']).agg({
    'Payments': 'sum',
    'Receipts': 'sum',
    'Balance': 'last',  # Carry forward the last balance
    'cleaned_particulars': 'first',  # You can change this depending on how to handle the description
    'Category': 'first'  # Assuming you want the first category for each day/cluster
}).reset_index()

# If there are no transactions for a day, the Payments and Receipts will remain 0
df_merged['Payments'].fillna(0, inplace=True)
df_merged['Receipts'].fillna(0, inplace=True)

# Finally, fill in the balance correctly, keeping it continuous if no transactions
df_merged['Balance'] = df_merged.groupby('Cluster')['Balance'].fillna(method='ffill')

# Output the resulting DataFrame
print(df_merged)


            Date  Cluster  Payments  Receipts       Balance  \
0     2022-11-06       -1       0.0       0.0  9.424640e+12   
1     2022-11-06        0    6030.0       0.0  1.245464e+13   
2     2022-11-06        1    3030.0       0.0  9.424640e+12   
3     2022-11-06        2       0.0       0.0  9.424640e+12   
4     2022-11-06        3       0.0       0.0  9.424640e+12   
...          ...      ...       ...       ...           ...   
10629 2025-01-31        7       0.0       0.0  1.037220e+12   
10630 2025-01-31        8       0.0       0.0  1.037220e+12   
10631 2025-01-31        9       0.0       0.0  1.037220e+12   
10632 2025-01-31       10       0.0       0.0  1.037220e+12   
10633 2025-01-31       11       0.0       0.0  1.037220e+12   

            cleaned_particulars        Category  
0                          None            None  
1              t ahirt othbnk t  NADIL OTHBNK T  
2      010001088282101 othbnk t        OTHBNK T  
3                          None            

C:\Users\isuru\AppData\Local\Temp\ipykernel_9936\1868295222.py:22: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_merged['Balance'] = df_merged['Balance'].fillna(method='ffill')
C:\Users\isuru\AppData\Local\Temp\ipykernel_9936\1868295222.py:34: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_merged['Payments'].fillna(0, inplace=True)
C:\Users\isuru\AppData\Local\Temp\ipykernel_9936\1868295222.py:35: FutureWarning: A value is trying to b

In [6]:
# Clean the Balance column (remove commas and handle 'B' for billions)
df['Balance'] = df['Balance'].replace({',': '', 'B': ''}, regex=True)

# If 'B' represents billions, convert the Balance to numeric, multiplying by 1e9 if 'B' was present
df['Balance'] = pd.to_numeric(df['Balance'], errors='coerce')

# Now, if any balance column is still represented in billions (assuming 'B' was not removed correctly)
df['Balance'] = df['Balance'] * 1e9  # If 'B' was not handled, this step multiplies by 1e9

# Ensure Balance is float type
df['Balance'] = df['Balance'].astype(float)

# Continue with the rest of your operations
